In [ ]:
Social media comments - Miniproject 5

In [ ]:
Step 1: installing all the necessary libraries and doing the pre-processing data cleaning.

In [5]:
%pip install numpy pandas matplotlib seaborn openpyxl tensorflow nltk scikit-learn

  Obtaining dependency information for numpy from https://files.pythonhosted.org/packages/c5/31/7fc6239c12bce7e931463251cca4426c465e1876ba3cc785402ef4dd8f4e/numpy-2.4.6-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for pandas from https://files.pythonhosted.org/packages/eb/62/c321f13b5ba1819fc8dca456c7fce578da2dcfecff1abbf0eaddf8406c0f/pandas-3.0.3-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for matplotlib from https://files.pythonhosted.org/packages/d0/9f/970fcbf381e82ec66fdf5da8ea76e2e9240f61a24011ce9fd1d42c37ac2d/matplotlib-3.11.0-cp311-cp311-win_amd64.whl.metadata
     ---------------------------------------- 0.0/80.3 kB ? eta -:--:--
     ----- ---------------------------------- 10.2/80.3 kB ? eta -:--:--
     --------------------------------- ---- 71.7/80.3 kB 660.6 kB/s eta 0:00:01
     -------------------------------------- 80.3/80.3 kB 642.4 kB/s eta 0:00:00
  Obtaining dependency information for seaborn from https://files.pyth


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
import os
import re
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.corpus import stopwords
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization
from transformers import BertTokenizer

nltk.download('stopwords', quiet=True)

sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 10, 'axes.labelsize': 12, 'axes.titlesize': 14})

STOP_WORDS = set(stopwords.words('english'))
NEGATIVES = {'not', 'no', 'nor', 'neither', 'never', 'against'}
CUSTOM_STOP_WORDS = STOP_WORDS.difference(NEGATIVES)

def load_raw_dataframes():
    train_path = r"C:\Users\pc\Desktop\miniproject5\train.xlsx"
    test_path = r"C:\Users\pc\Desktop\miniproject5\test.xlsx"
    
    print(f"[1.1] Extracting records from: {train_path}")
    train_df = pd.read_excel(train_path)
    
    print(f"[1.2] Extracting records from: {test_path}")
    test_df = pd.read_excel(test_path)
    
    train_df['comment_text'] = train_df['comment_text'].fillna("").astype(str)
    test_df['comment_text'] = test_df['comment_text'].fillna("").astype(str)
    
    return train_df, test_df

train_df, test_df = load_raw_dataframes()

train_df['word_count'] = train_df['comment_text'].apply(lambda x: len(x.split()))
p95_words = np.percentile(train_df['word_count'], 95)
MAX_SEQUENCE_LENGTH = int(np.ceil(p95_words))
print(f"CRITICAL VALUE: RNN/LSTM MAX_SEQUENCE_LENGTH set to: {MAX_SEQUENCE_LENGTH}")


def clean_for_rnn_lstm(text):
    """Cleans text for traditional RNN/LSTM models while preserving key toxic sentiment signals."""
    text = str(text).lower()
    
    text = re.sub(r"can't", "cannot", text)
    text = re.sub(r"n't", " not", text)
    text = re.sub(r"i'm", "i am", text)
    text = re.sub(r"\'re", " are", text)
    text = re.sub(r"\'s", " is", text)
    text = re.sub(r"\'ll", " will", text)
    text = re.sub(r"\'ve", " have", text)
    
    text = re.sub(r"[^a-zA-Z!?\s]", "", text)
    
    words = text.split()
    # Uses the global optimized set
    filtered_words = [w for w in words if w not in CUSTOM_STOP_WORDS]
    return " ".join(filtered_words)

print("[3.1] Generating cleaned tokens for RNN/LSTM models...")
train_df['cleaned_rnn_text'] = train_df['comment_text'].apply(clean_for_rnn_lstm)
test_df['cleaned_rnn_text'] = test_df['comment_text'].apply(clean_for_rnn_lstm)

train_df['bert_text'] = train_df['comment_text']
test_df['bert_text'] = test_df['comment_text']

target_cols = [col for col in ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate'] if col in train_df.columns]
if not target_cols:
    target_cols = ['toxic']

y_train = train_df[target_cols].values
y_test = test_df[target_cols].values if all(col in test_df.columns for col in target_cols) else None

MAX_FEATURES = 5000  

vectorizer = TextVectorization(
    max_tokens=MAX_FEATURES,
    standardize=None,
    output_mode='int',
    output_sequence_length=MAX_SEQUENCE_LENGTH
)

print("[4.1] Fitting TextVectorization layer ONLY on Train text vocabulary...")
vectorizer.adapt(train_df['cleaned_rnn_text'].values)

X_train_rnn = vectorizer(train_df['cleaned_rnn_text'].values).numpy()
X_test_rnn = vectorizer(test_df['cleaned_rnn_text'].values).numpy()

print(f"RNN Feature Tensors Ready. Shape: {X_train_rnn.shape}")

print("[5.1] Initializing BERT Tokenizer and generating token embeddings...")
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def encode_bert_data(text_series, max_len=128, batch_size=256):
    texts = text_series.tolist()
    all_input_ids = []
    all_attention_masks = []
    
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        encoded = tokenizer(
            text=batch_texts,
            add_special_tokens=True,
            max_length=max_len, 
            padding='max_length',
            truncation=True,
            return_tensors='np'
        )
        all_input_ids.append(encoded['input_ids'])
        all_attention_masks.append(encoded['attention_mask'])
        
    return {
        'input_ids': np.vstack(all_input_ids),
        'attention_mask': np.vstack(all_attention_masks)
    }

X_train_bert = encode_bert_data(train_df['bert_text'])
X_test_bert = encode_bert_data(test_df['bert_text'])

print(f"BERT Inputs Loaded. Input ID shape: {X_train_bert['input_ids'].shape}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

main_target = target_cols[0]
sns.countplot(data=train_df, x=main_target, hue=main_target, palette='coolwarm', ax=axes[0], legend=False)
axes[0].set_title(f"Training Target Class Distribution ({main_target})")
axes[0].set_xlabel("Class Designation")
axes[0].set_ylabel("Total Samples")

for cls, color, lbl in zip([0, 1], ['teal', 'crimson'], ['Clean Content', 'Toxic Content']):
    subset = train_df[train_df[main_target] == cls]
    if not subset.empty:
        sns.kdeplot(subset['word_count'], fill=True, color=color, label=lbl, ax=axes[1], clip=(0, None))
axes[1].set_title("Raw Sequence Word Count Profile")
axes[1].set_xlabel("Word Length Per Entry")
axes[1].set_ylabel("Kernel Density")
axes[1].legend()

plt.tight_layout()
plt.savefig('eda_dashboard_metrics.png', dpi=150)
print("[6.0] SUCCESS: Pipeline processing complete. Graphic diagnostic metrics exported.")

[1.1] Extracting records from: C:\Users\pc\Desktop\miniproject5\train.xlsx


KeyboardInterrupt: 

In [ ]:
Step 2: Model development(LSTM, RNN and BERT architectures)

In [ ]:
BERT step. Transformer-based models (like BERT, RoBERTa, or DistilBERT) are decisively the best architecture for toxicity detection.
While RNNs and LSTMs process text word-by-word sequentially, Transformers utilize a mechanism called Self-Attention. This allows the model to look at every word in a comment simultaneously and evaluate its relationship to every other word.
Because toxic language relies heavily on subtle context, sarcasm, and long-range dependencies (e.g., words separated by a long sentence), Transformers capture these dynamics significantly better, yielding far higher accuracy and lower false-positive rates.

In [ ]:
Checking which model is best and storing it for deployment.

In [12]:
%pip install transformers huggingface_hub

  Obtaining dependency information for transformers from https://files.pythonhosted.org/packages/29/47/54eacf96b5c835bbd6ca631aa2740e7705ed63d9e3a8afd2d2cc6d09cae5/transformers-5.13.1-py3-none-any.whl.metadata
  Obtaining dependency information for huggingface_hub from https://files.pythonhosted.org/packages/f1/ce/13b2ba57838b8db1e6bd033c1b21ce0b9f6153b87d4e4939f77074e41eb0/huggingface_hub-1.23.0-py3-none-any.whl.metadata
  Obtaining dependency information for pyyaml>=5.1 from https://files.pythonhosted.org/packages/da/e3/ea007450a105ae919a72393cb06f122f288ef60bba2dc64b26e2646fa315/pyyaml-6.0.3-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for tokenizers<=0.23.0,>=0.22.0 from https://files.pythonhosted.org/packages/65/71/0670843133a43d43070abeb1949abfdef12a86d490bea9cd9e18e37c5ff7/tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata
  Obtaining dependency information for typer from https://files.pythonhosted.org/packages/80/87/b9fd69c92c6102a066e1b86a35243f53e70bd


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
pip install numpy pandas openpyxl tensorflow transformers scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
%pip install tensorflow transformers huggingface_hub openpyxl scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
RNN and LSTM experimentation and evaluation.

In [32]:
import os
import re
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization, Embedding, SimpleRNN, LSTM, Bidirectional, GlobalMaxPool1D, Dropout, Dense
from sklearn.metrics import classification_report, roc_auc_score

LABEL_COLUMNS = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
MAX_FEATURES = 5000
MAX_LEN = 128
BATCH_SIZE = 32

print("Ingesting train and test dataset configurations...")
train_df = pd.read_excel(r"C:\Users\pc\Desktop\miniproject5\train.xlsx")
for col in LABEL_COLUMNS:
    train_df[col] = train_df[col].fillna(0).astype(int)
train_df['comment_text'] = train_df['comment_text'].fillna("").astype(str)

test_df = pd.read_excel(r"C:\Users\pc\Desktop\miniproject5\test.xlsx")
test_df['comment_text'] = test_df['comment_text'].fillna("").astype(str)

def basic_clean(text):
    text = str(text).lower()
    text = re.sub(r"https?://\S+|www\.\S+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

train_df['comment_text'] = train_df['comment_text'].apply(basic_clean)
test_df['comment_text'] = test_df['comment_text'].apply(basic_clean)

X_train_raw = train_df['comment_text'].values
X_test_raw = test_df['comment_text'].values
y_train = train_df[LABEL_COLUMNS].values

has_test_labels = set(LABEL_COLUMNS).issubset(test_df.columns)
if has_test_labels:
    for col in LABEL_COLUMNS:
        test_df[col] = test_df[col].fillna(0).astype(int)
    y_test = test_df[LABEL_COLUMNS].values
else:
    y_test = None

def evaluate_multi_label_performance(model_name, y_true, y_pred_probs):
    print(f"\n" + "="*60 + f"\n📊 {model_name} COMPREHENSIVE PERFORMANCE REPORT\n" + "="*60)
    
    y_pred_binary = (y_pred_probs >= 0.5).astype(int)
    print("--- Core Classification Matrix Breakout (F1, Precision, Recall) ---")
    print(classification_report(y_true, y_pred_binary, target_names=LABEL_COLUMNS, zero_division=0))
    
    print("--- Per-Label ROC-AUC Scores ---")
    individual_auc_scores = []
    for idx, label_name in enumerate(LABEL_COLUMNS):
        try:
            if len(np.unique(y_true[:, idx])) > 1:
                auc = roc_auc_score(y_true[:, idx], y_pred_probs[:, idx])
                print(f" * ROC-AUC ({label_name:13}): {auc:.4f}")
                individual_auc_scores.append(auc)
            else:
                print(f" * ROC-AUC ({label_name:13}): Skipping (No positive cases in test split)")
        except Exception as e:
            print(f" * ROC-AUC ({label_name:13}): Metric error -> {e}")
            
    if individual_auc_scores:
        macro_auc = np.mean(individual_auc_scores)
        print(f"\n * Global Mean ROC-AUC Score: {macro_auc:.4f}")

vectorizer = TextVectorization(max_tokens=MAX_FEATURES, output_mode='int', output_sequence_length=MAX_LEN)
vectorizer.adapt(X_train_raw)

X_train_keras = vectorizer(X_train_raw).numpy()
X_test_keras = vectorizer(X_test_raw).numpy()

vocab = vectorizer.get_vocabulary()
with open('vectorizer_vocab.pkl', 'wb') as f:
    pickle.dump({'vocab': vocab, 'max_len': MAX_LEN, 'max_features': MAX_FEATURES}, f)
print("SUCCESS: Vectorizer components natively serialized.")

print("\n" + "="*50 + "\nMODEL 1: VANILLA RNN TRAINING & TESTING\n" + "="*50)
def build_rnn():
    model = tf.keras.Sequential([
        Embedding(input_dim=MAX_FEATURES, output_dim=64, mask_zero=False),
        SimpleRNN(32, return_sequences=True), 
        GlobalMaxPool1D(),
        Dropout(0.2),
        Dense(64, activation='relu'),
        Dense(len(LABEL_COLUMNS), activation='sigmoid')
    ], name="Vanilla_RNN")
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
    return model

rnn_model = build_rnn()
rnn_model.fit(X_train_keras, y_train, batch_size=BATCH_SIZE, epochs=2, validation_split=0.1, verbose=1)
rnn_model.save_weights('rnn_toxicity_model.weights.h5')

rnn_preds = rnn_model.predict(X_test_keras)
if has_test_labels:
    evaluate_multi_label_performance("Vanilla RNN", y_test, rnn_preds)

print("\n" + "="*50 + "\nMODEL 2: BIDIRECTIONAL LSTM TRAINING & TESTING\n" + "="*50)
def build_lstm():
    model = tf.keras.Sequential([
        Embedding(input_dim=MAX_FEATURES, output_dim=64, mask_zero=False),
        Bidirectional(LSTM(32, return_sequences=True)), 
        GlobalMaxPool1D(),
        Dropout(0.2),
        Dense(64, activation='relu'),
        Dense(len(LABEL_COLUMNS), activation='sigmoid')
    ], name="Bidirectional_LSTM")
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
    return model

lstm_model = build_lstm()
lstm_model.fit(X_train_keras, y_train, batch_size=BATCH_SIZE, epochs=2, validation_split=0.1, verbose=1)
lstm_model.save_weights('lstm_toxicity_model.weights.h5')

lstm_preds = lstm_model.predict(X_test_keras)
if has_test_labels:
    evaluate_multi_label_performance("Bidirectional LSTM", y_test, lstm_preds)

Ingesting train and test dataset configurations...


KeyboardInterrupt: 

In [26]:
pip install --upgrade tensorflow transformers h5py

  Obtaining dependency information for h5py from https://files.pythonhosted.org/packages/f7/20/e6c0ff62ca2ad1a396a34f4380bafccaaf8791ff8fccf3d995a1fc12d417/h5py-3.16.0-cp311-cp311-win_amd64.whl.metadata
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
BERT model experimentation and evaluation.

In [13]:
import sys
!{sys.executable} -m pip install --upgrade pip
!{sys.executable} -m pip install --upgrade "transformers[tf-cpu]" tf-keras tensorflow

  Obtaining dependency information for pip from https://files.pythonhosted.org/packages/5d/95/6b5cb3461ea5673ba0995989746db58eb18b91b54dbf331e72f569540946/pip-26.1.2-py3-none-any.whl.metadata
  Using cached pip-26.1.2-py3-none-any.whl.metadata (4.6 kB)
Using cached pip-26.1.2-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 23.2.1
    Uninstalling pip-23.2.1:
      Successfully uninstalled pip-23.2.1
  Using cached tf_keras-2.21.0-py3-none-any.whl.metadata (1.8 kB)
Using cached tf_keras-2.21.0-py3-none-any.whl (1.7 MB)


In [21]:
import sys
!{sys.executable} -m pip install --upgrade pip
!{sys.executable} -m pip install tensorflow-hub tensorflow-text pandas openpyxl scikit-learn

ERROR: Could not find a version that satisfies the requirement tensorflow-text (from versions: none)
ERROR: No matching distribution found for tensorflow-text


In [ ]:
BERT experimentation and evaluation.

In [33]:
import os
import gc
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import Dropout, Dense, Input, GlobalAveragePooling1D
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report 

print("Validating environment subsystems...")
try:
    from transformers import DistilBertTokenizerFast, TFDistilBertModel
    print("✅ Advanced BERT Processing Infrastructure Online.")
    USE_BERT = True
except ImportError:
    print("⚠️ Specialized Transformers not found. Using Native TF Workflow...")
    USE_BERT = False

print("\n[Step 1/3] Loading training and testing Excel spreadsheets...")
df_train_raw = pd.read_excel(r"C:\Users\pc\Desktop\miniproject5\train.xlsx")
df_test_raw = pd.read_excel(r"C:\Users\pc\Desktop\miniproject5\test.xlsx")

TEXT_COLUMN = "comment_text"
LABEL_COLUMNS = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

df_train = df_train_raw[[TEXT_COLUMN] + LABEL_COLUMNS].dropna().reset_index(drop=True)
df_test = df_test_raw[[TEXT_COLUMN]].dropna().reset_index(drop=True)

print(f"Dataset sizes - Train: {len(df_train)} rows | Test: {len(df_test)} rows")

X_train_raw = df_train[TEXT_COLUMN].astype(str).tolist()
y_all = df_train[LABEL_COLUMNS].values.astype("float32")
X_test_raw = df_test[TEXT_COLUMN].astype(str).tolist()

X_train, X_val, y_train, y_val = train_test_split(X_train_raw, y_all, test_size=0.15, random_state=42)

del df_train_raw, df_test_raw, df_train, df_test, X_train_raw, y_all
gc.collect()

print("Successfully partitioned text columns into Python lists without RAM spikes!")

print("\n[Step 2/3] Initializing Deep Learning Model Architecture...")

MAX_LEN = 64

if USE_BERT:
    tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
    bert_backbone = TFDistilBertModel.from_pretrained("distilbert-base-uncased")
    bert_backbone.trainable = False

    def encode_data_batched(texts, batch_size=10000):
        """Processes text lists in chunks to keep memory flat"""
        all_ids, all_masks = [], []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            encodings = tokenizer(batch, padding="max_length", truncation=True, max_length=MAX_LEN, return_tensors="np")
            all_ids.append(encodings["input_ids"])
            all_masks.append(encodings["attention_mask"])
        return np.vstack(all_ids), np.vstack(all_masks)

    print("Encoding text data via batched memory pipelines...")
    X_train_ids, X_train_mask = encode_data_batched(X_train)
    X_val_ids, X_val_mask = encode_data_batched(X_val)
    X_test_ids, X_test_mask = encode_data_batched(X_test_raw)

    tf_train_inputs = [X_train_ids, X_train_mask]
    tf_val_inputs = [X_val_ids, X_val_mask]
    tf_test_inputs = [X_test_ids, X_test_mask]

    del X_train, X_val
    gc.collect()

    input_ids = Input(shape=(MAX_LEN,), dtype=tf.int32, name="input_ids")
    attention_mask = Input(shape=(MAX_LEN,), dtype=tf.int32, name="attention_mask")
    
    transformer_outputs = bert_backbone(input_ids=input_ids, attention_mask=attention_mask)
    sequence_output = transformer_outputs.last_hidden_state
    pooling_layer = GlobalAveragePooling1D()(sequence_output)
    
else:
    MAX_VOCAB = 20000
    vectorization_layer = tf.keras.layers.TextVectorization(max_tokens=MAX_VOCAB, output_sequence_length=MAX_LEN, output_mode="int")
    vectorization_layer.adapt(X_train)

    tf_train_inputs = tf.constant(X_train, dtype=tf.string)
    tf_val_inputs = tf.constant(X_val, dtype=tf.string)
    tf_test_inputs = tf.constant(X_test_raw, dtype=tf.string)

    string_input = Input(shape=(), dtype=tf.string, name="string_input")
    vectors = vectorization_layer(string_input)
    embeddings = tf.keras.layers.Embedding(input_dim=MAX_VOCAB, output_dim=128)(vectors)
    
    pooling_layer = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32))(embeddings)

dropout_layer = Dropout(0.3)(pooling_layer)
dense_hidden = Dense(64, activation="relu")(dropout_layer)
output_layer = Dense(len(LABEL_COLUMNS), activation="sigmoid", name="toxicity_outputs")(dense_hidden)

if USE_BERT:
    dl_model = tf.keras.Model(inputs=[input_ids, attention_mask], outputs=output_layer)
else:
    dl_model = tf.keras.Model(inputs=string_input, outputs=output_layer)

dl_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5 if USE_BERT else 1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(multi_label=True, name="AUC")]
)

callbacks_list = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)
]

print("\n--- Model Training Pipeline Executing ---")
dl_model.fit(
    tf_train_inputs, y_train,
    validation_data=(tf_val_inputs, y_val),
    batch_size=32, 
    epochs=3,
    callbacks=callbacks_list
)

if not USE_BERT:
    del X_train, X_val
    gc.collect()

print("\n[Step 3/3] Commencing Final System Evaluations...")

print("Evaluating Multi-Label metrics across validation matrices...")
val_probabilities = dl_model.predict(tf_val_inputs, batch_size=32)
val_predictions_binary = (val_probabilities > 0.5).astype(int)

print("\n" + "="*65 + "\n    MANDATORY PROJECT DISCOVERY MATRICES (VAL DATASET EVALUATION)\n" + "="*65)
print(classification_report(
    y_val, 
    val_predictions_binary, 
    target_names=LABEL_COLUMNS,
    zero_division=0
))
print("="*65 + "\n")

print("Generating target probabilities across holdout matrix...")
predictions = dl_model.predict(tf_test_inputs, batch_size=32)

df_output = pd.DataFrame(X_test_raw, columns=[TEXT_COLUMN])
for idx, label in enumerate(LABEL_COLUMNS):
    df_output[label] = predictions[:, idx]

output_filename = "test_predictions_final.xlsx"
df_output.to_excel(output_filename, index=False)
print(f"📊 Evaluation results saved to file: '{output_filename}'")

model_storage_name = "saved_toxicity_model.keras" if USE_BERT else "saved_toxicity_model_tf"
dl_model.save(model_storage_name)
print(f"💾 Permanent Deployment Model saved to: '{model_storage_name}'")
print("\n✅ Model Development Milestone Complete with full metrics integration.")

Validating environment subsystems...
⚠️ Specialized Transformers not found. Using Native TF Workflow...

[Step 1/3] Loading training and testing Excel spreadsheets...
Dataset sizes - Train: 159449 rows | Test: 153038 rows
Successfully partitioned text columns into Python lists without RAM spikes!

[Step 2/3] Initializing Deep Learning Model Architecture...

--- Model Training Pipeline Executing ---
Epoch 1/3
 450/4236 ━━━━━━━━━━━━━━━━━━━━ 16:02 254ms/step - AUC: 0.6839 - accuracy: 0.8903 - loss: 0.1461

KeyboardInterrupt: 